# NOTEBOOK 04 – TRÚC CHI | CHƯƠNG 4: MÔ HÌNH & THỰC NGHIỆM

**Phụ trách:** 4.1–4.5 · Issue **#5** · Hạn báo cáo/Validation: 22/11/2026; bàn giao dự đoán Test cho Vĩnh Chi trước 24/11.

**Chỉ huấn luyện:** Dummy Classifier và MLP. **Tiếp nhận RF** Ngọc Lan đã huấn luyện, không fit lại Random Forest.

**Đầu vào** từ Lam Linh: `data/processed/orders_model_ready.csv`. **Bàn giao từ Ngọc Lan:** `models/ngoc_lan_random_forest.joblib`, `data/processed/rf_val_predictions.csv`, `rf_metadata.json` dùng cùng split.

**Đầu ra**: `test_predictions.csv`, `experiment_metadata.json`, các model `joblib` (không push), bảng tham số và ngưỡng trên Validation.

[Lý thuyết Ch4](../docs/theory/04_truc_chi_ch4.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

> Chạy **chỉ sau khi Ch3 bàn giao** và audit. Không đưa thông tin tương lai vào X. Không báo kết quả trước khi chạy; các cell ở đây là code gợi ý chuẩn hóa để hoàn thiện.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 4.1 Cố định chia Train/Validation/Test để dùng CHUNG

**Trúc Chi là người phụ trách quy ước split:** chia theo `order_approved_at`, kiểm soát trưởng thành nhãn, lưu split manifest + fingerprint. Ngọc Lan sử dụng `src.experiment.get_shared_splits` để huấn luyện RF theo đúng các chỉ mục giống MLP và Dummy.

**Quy trình phối hợp:** Ch3 bàn giao dữ liệu → Trúc Chi xác nhận split → Ngọc Lan huấn luyện RF (Ch2) → Trúc Chi tiếp nhận RF, fit Dummy/MLP và tích hợp → Vĩnh Chi kiểm tra ba mô hình trên Test.

**Không xáo lại dữ liệu hoặc split riêng cho từng thành viên.** Tất cả model dùng chung `SAFE_FEATURES`, train/val/test và nhãn của Lam Linh.

In [ ]:
from src.experiment import (get_shared_splits, NUMERIC_FEATURES, CATEGORY_FEATURES,
                            SAFE_FEATURES, choose_threshold_on_validation)
splits = get_shared_splits(PROCESSED)
train, val, test = splits["train"], splits["val"], splits["test"]
split_info = splits["metadata"]
split_id = split_info["split_id"]
FEATURES = list(SAFE_FEATURES)
print("Shared split id:", split_id)
display(pd.DataFrame([
    {"split":label, "n":len(splits[label]),
     "late_rate":splits[label]["is_late"].mean()}
    for label in ("train","val","test")
]))
print("Ngọc Lan và Trúc Chi phải dùng cùng split ID; model RF không được fit lại.")

## 4.2 Pipeline và Dummy baseline – Trúc Chi trực tiếp huấn luyện

Fit preprocessing **trên Train**; impute/one-hot categorical, scale numeric riêng cho MLP. Feature whitelist thống nhất ở `src/experiment.py`.

- Trúc Chi huấn luyện **Dummy (tham chiếu)** và **MLP (Deep Learning)**.
- **Không huấn luyện Random Forest trong notebook 04.** RF phải được đọc từ artifact `models/ngoc_lan_random_forest.joblib` của Ngọc Lan.
- Nếu thiếu artifact RF thì dừng tại 4.3 để Ngọc Lan bàn giao, không tự tạo model RF thay thế.

In [ ]:
import json
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score

NUMERIC = list(NUMERIC_FEATURES)
CATEGORICAL = list(CATEGORY_FEATURES)
X_train, y_train = train[FEATURES], train["is_late"].astype(int)
X_val, y_val = val[FEATURES], val["is_late"].astype(int)
X_test, y_test = test[FEATURES], test["is_late"].astype(int)
assert y_train.nunique() == 2 and y_val.nunique() == 2

def make_preprocessor(scale_numeric=False):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(steps), NUMERIC),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORICAL)
    ])

# Trúc Chi CHỈ fit 2 mô hình này:
models = {
    "dummy": Pipeline([
        ("prep", make_preprocessor(False)),
        ("model", DummyClassifier(strategy="prior"))
    ]),
    "mlp": Pipeline([
        ("prep", make_preprocessor(True)),
        ("model", MLPClassifier(hidden_layer_sizes=(64,32), alpha=0.001,
                                early_stopping=True, max_iter=180,
                                random_state=42))
    ]),
}
print("Các mô hình Trúc Chi được giao huấn luyện:", list(models))

## 4.3 Tiếp nhận & xác minh Random Forest của Ngọc Lan; 4.4 Huấn luyện Dummy/MLP

### 4.3 Random Forest được Ngọc Lan huấn luyện ở Chương 2
Trúc Chi **không gọi `fit` cho RF**, mà kiểm tra đầy đủ: model artifact, metadata, feature whitelist, `split_id`, thứ tự `order_id` trong Validation, và xác suất từ `predict_proba`.

### 4.4 Huấn luyện các mô hình còn lại
Trúc Chi trực tiếp `fit` Dummy và MLP trên Train, tính AP trên Validation. Nếu MLP có lỗi hội tụ, cần kiểm tra learning curve và tham số; không sửa theo kết quả Test.

In [ ]:
from sklearn.metrics import average_precision_score

rf_path = ROOT / "models" / "ngoc_lan_random_forest.joblib"
rf_meta_path = PROCESSED / "rf_metadata.json"
rf_val_path = PROCESSED / "rf_val_predictions.csv"
missing = [p.name for p in (rf_path, rf_meta_path, rf_val_path) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Chưa nhận RF từ Ngọc Lan: " + ", ".join(missing)
        + ". Yêu cầu bàn giao artifact qua kênh nội bộ nhóm."
    )
rf_meta = json.loads(rf_meta_path.read_text(encoding="utf-8"))
rf_val_file = pd.read_csv(rf_val_path)
assert rf_meta["owner"] == "Ngọc Lan" and rf_meta["model"] == "rf"
assert rf_meta["split_id"] == split_id, "Ngọc Lan và Trúc Chi đã dùng split khác nhau!"
assert rf_meta["dataset_sha256"] == split_info["csv_sha256"]
assert rf_meta["features"] == FEATURES
assert rf_meta["validation_n"] == len(val)
assert np.array_equal(rf_val_file["order_id"].to_numpy(), val["order_id"].to_numpy())
assert np.array_equal(rf_val_file["y_true"].to_numpy(), y_val.to_numpy())

rf = joblib.load(rf_path)  # chỉ load artifact từ thành viên nhóm đã xác minh
p_rf_verified = rf.predict_proba(X_val)[:, 1]
assert np.allclose(p_rf_verified, rf_val_file["p_rf"].to_numpy(), atol=1e-7)
models["rf"] = rf  # CHỈ nạp, KHÔNG fit RF

validation_prob = {"rf": p_rf_verified}
validation_results = [
    {"model": "rf", "owner": "Ngọc Lan",
     "val_AP": float(average_precision_score(y_val, p_rf_verified))}
]
for name in ("dummy", "mlp"):
    model = models[name]
    model.fit(X_train, y_train)  # Trúc Chi fit Dummy/MLP trên Train chung
    probs = model.predict_proba(X_val)[:, 1]
    validation_prob[name] = probs
    validation_results.append({
        "model": name, "owner": "Trúc Chi",
        "val_AP": float(average_precision_score(y_val, probs))
    })
val_scores = pd.DataFrame(validation_results).sort_values("val_AP", ascending=False)
display(val_scores)
print("RF được xác minh, Dummy/MLP đã huấn luyện trên chung Train/Validation.")

## 4.5 Chọn mô hình, ngưỡng và Top-K chỉ bằng Validation

- Ngọc Lan đã chọn threshold **RF bằng Validation** và bàn giao trong `rf_metadata.json`. Trúc Chi dùng lại chính xác ngưỡng đó.
- Trúc Chi chọn ngưỡng Dummy và MLP bằng **F1 Validation**, đồng thời chọn champion giữa RF và MLP bằng AP Validation. Dummy là baseline tham chiếu.
- Ngưỡng/champion được **đóng băng** trước khi sinh dự đoán Test. Sau đó **Vĩnh Chi đánh giá độc lập tất cả mô hình**.

Không thay ngưỡng hoặc siêu tham số theo Test.

In [ ]:
from src.experiment import classification_metrics
thresholds = {
    "rf": float(rf_meta["validation_metrics"]["threshold"]),
}
val_metric_rows = []
for name in ("dummy", "mlp"):
    record = choose_threshold_on_validation(y_val, validation_prob[name])
    thresholds[name] = float(record["threshold"])
for name in ("dummy", "rf", "mlp"):
    result = classification_metrics(y_val, validation_prob[name], thresholds[name])
    val_metric_rows.append({"model": name,
                            "owner": "Ngọc Lan" if name == "rf" else "Trúc Chi",
                            **result, "dataset": "Validation"})
if not np.isclose(val_metric_rows[1]["AP"],
                  float(rf_meta["validation_metrics"]["AP"]), atol=1e-8):
    raise ValueError("AP RF trên Validation không khớp file Ngọc Lan.")
display(pd.DataFrame(val_metric_rows))

champion = max((r for r in validation_results if r["model"] in ("rf","mlp")),
               key=lambda r: r["val_AP"])["model"]
chosen_threshold = thresholds[champion]
print("Champion dựa trên Validation:", champion, "| threshold:", chosen_threshold)
print("Không dùng dữ liệu Test để chọn ngưỡng.")

## 4.6 Đóng băng frozen Test predictions và bàn giao Vĩnh Chi

**Chỉ chạy khi** RF đã được xác minh và Dummy/MLP được huấn luyện trên **cùng Train**; model/threshold lựa chọn theo Validation.

Xuất `test_predictions.csv` gồm `order_id, y_true, p_dummy, p_rf, p_mlp, y_pred_champion`. Metadata có nguồn gốc RF từ Ngọc Lan và `split_id` do Ch4 chuẩn hóa. Vĩnh Chi tính lại mọi metrics/Test; Ch4 không điều chỉnh dựa trên Test.

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
test_probabilities = {
    name: models[name].predict_proba(X_test)[:, 1]
    for name in ("dummy", "rf", "mlp")
}
predictions = pd.DataFrame({
    "order_id": test["order_id"].to_numpy(),
    "y_true": y_test.to_numpy(),
    "p_dummy": test_probabilities["dummy"],
    "p_rf": test_probabilities["rf"],
    "p_mlp": test_probabilities["mlp"],
})
predictions["y_pred_champion"] = (
    predictions["p_" + champion].to_numpy() >= chosen_threshold
).astype(int)
assert len(predictions) == len(test) and predictions["order_id"].is_unique
predictions.to_csv(PROCESSED / "test_predictions.csv", index=False)
joblib.dump(models["dummy"], MODEL_DIR / "dummy_truc_chi.joblib")
joblib.dump(models["mlp"], MODEL_DIR / "mlp_truc_chi.joblib")
joblib.dump(models[champion], MODEL_DIR / "champion.joblib")
meta = {
    "prediction_time": "order_approved_at",
    "features": FEATURES,
    "split_id": split_id,
    "split_metadata": split_info,
    "model_owners": {"rf": "Ngọc Lan", "dummy": "Trúc Chi", "mlp": "Trúc Chi"},
    "rf_source": "models/ngoc_lan_random_forest.joblib",
    "champion": champion,
    "thresholds_from_val": thresholds,
    "chosen_threshold": chosen_threshold,
    "validation_AP": {
        r["model"]: r["val_AP"] for r in validation_results
    },
    "test_n": len(test),
    "random_state": 42,
    "label": "DATE(actual)>DATE(estimated) on delivered labeled orders",
}
(PROCESSED / "experiment_metadata.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã xuất frozen Test predictions, model champion và metadata.")
print("Bàn giao cho Vĩnh Chi (Ch5), KHÔNG dùng Test để tune.")

## 4.7 Checklist báo cáo & bàn giao
- [ ] Temporal split không trùng order_id, kiểm soát thời điểm trưởng thành nhãn.
- [ ] Có bảng train/val/test, tỷ lệ trễ từng tập, cutoff và số loại do chưa đủ nhãn.
- [ ] Preprocessing chỉ fit Train; cùng split cho RF/MLP/Dummy.
- [ ] Có log cấu hình, kết quả Validation, ngưỡng chốt.
- [ ] File Test predictions, Dummy/MLP fitted models và champion model có đủ metadata.
- [ ] Có lời giải thích tránh Data Leakage và giới hạn delivered-only.
- [ ] Ch5 đã xác nhận nhận file; đã hoàn thành `reports/chapters/CH04_MODELING.md`.

**Hướng dẫn trình bày:** mô tả setup → phương pháp → tham số → Validation → tiêu chí chọn mô hình → artifact bàn giao; không công bố kết quả Test làm căn cứ chọn model.
- [ ] Đã kiểm tra RF model Ngọc Lan và cùng split_id, không gọi fit cho RF.
- [ ] RF trên Validation xuất từ Ngọc Lan; Vĩnh Chi nhận frozen Test đánh giá cuối.